# ARC-AGI-2 Phase 1C symbolic solver

This notebook is the complete Phase 1C submission runtime. It uses only the Python standard library, a finite depth-two symbolic grammar, strict validation, and bounded deterministic candidate search. The default mode is `submission`; local scoring modes are selected with `ARC_MODE=evaluation` or `ARC_MODE=training`.

Submission mode reads only `arc-agi_test_challenges.json`, discovers task IDs and test counts from that file, validates two attempts per output, and writes `/kaggle/working/submission.json` on Kaggle. Local runs use the repository-root `artifacts/submission.json` unless `ARC_OUTPUT_PATH` is set. Set `ARC_SOLVER_MODE=identity` to reproduce the CPU baseline exactly.

The solver learns shared parameters from demonstrations only, validates exact candidates against every demonstration, and keeps distinct outputs with a valid identity/shape fallback. Training mode reports exact output/task metrics, demo coverage, bounded task timings, a deterministic hash split, and representative failures. The official evaluation is output-weighted: the highest score among the two attempts is taken for each test output, then divided by the total number of test outputs. See the [Kaggle evaluation overview](https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-2/overview/evaluation).

In [ ]:
from __future__ import annotations

import copy
import hashlib
import json
import os
import re
import time
from pathlib import Path
from typing import Any

MAX_GRID_SIDE = 30
TASK_ID_PATTERN = re.compile(r"[0-9a-f]{8}")


class ValidationError(ValueError):
    pass


def _require_exact_keys(value, expected, label):
    if type(value) is not dict or set(value) != set(expected):
        raise ValidationError(f"{label} must have exactly these keys: {expected}")


def validate_task_id(task_id):
    if type(task_id) is not str or TASK_ID_PATTERN.fullmatch(task_id) is None:
        raise ValidationError(f"invalid task ID: {task_id!r}")


def validate_grid(grid, label="grid"):
    if type(grid) is not list or not grid or len(grid) > MAX_GRID_SIDE:
        raise ValidationError(f"{label} must be a nonempty list of rows")
    width = None
    for row_number, row in enumerate(grid):
        if type(row) is not list or not row or len(row) > MAX_GRID_SIDE:
            raise ValidationError(f"{label} row {row_number} is malformed")
        if width is None:
            width = len(row)
        elif len(row) != width:
            raise ValidationError(f"{label} is not rectangular")
        for color in row:
            if type(color) is not int or not 0 <= color <= 9:
                raise ValidationError(f"{label} contains an invalid color")
    return len(grid), width


def validate_challenges(challenges):
    if type(challenges) is not dict or not challenges:
        raise ValidationError("challenges must be a nonempty object")
    for task_id, task in challenges.items():
        validate_task_id(task_id)
        _require_exact_keys(task, ("train", "test"), f"task {task_id}")
        if type(task["train"]) is not list or not task["train"]:
            raise ValidationError(f"task {task_id} has malformed train data")
        for pair_number, pair in enumerate(task["train"]):
            _require_exact_keys(pair, ("input", "output"), f"train pair {pair_number}")
            validate_grid(pair["input"], f"task {task_id} train input")
            validate_grid(pair["output"], f"task {task_id} train output")
        if type(task["test"]) is not list or not task["test"]:
            raise ValidationError(f"task {task_id} has malformed test data")
        for test_number, test in enumerate(task["test"]):
            _require_exact_keys(test, ("input",), f"test input {test_number}")
            validate_grid(test["input"], f"task {task_id} test input")


def validate_solutions(challenges, solutions):
    if type(solutions) is not dict or set(solutions) != set(challenges):
        raise ValidationError("solutions must contain exactly the challenge task IDs")
    for task_id, task in challenges.items():
        outputs = solutions[task_id]
        if type(outputs) is not list or len(outputs) != len(task["test"]):
            raise ValidationError(f"solution count mismatch for task {task_id}")
        for output_number, output in enumerate(outputs):
            validate_grid(output, f"task {task_id} solution {output_number}")


def validate_submission(challenges: dict[str, Any], submission: dict[str, Any]) -> None:
    validate_challenges(challenges)
    if type(submission) is not dict or set(submission) != set(challenges):
        raise ValidationError("submission must contain exactly the challenge task IDs")
    for task_id, task in challenges.items():
        predictions: list[dict[str, Any]] = submission[task_id]
        if type(predictions) is not list or len(predictions) != len(task["test"]):
            raise ValidationError(f"prediction count mismatch for task {task_id}")
        for output_number, prediction in enumerate(predictions):
            _require_exact_keys(
                prediction, ("attempt_1", "attempt_2"), f"prediction {output_number}"
            )
            validate_grid(prediction["attempt_1"], f"task {task_id} attempt_1")
            validate_grid(prediction["attempt_2"], f"task {task_id} attempt_2")


# ceiling: 1024 programs per task; depth-one programs always come first.
MAX_PROGRAMS_PER_TASK = 1024
MAX_COMPOSITION_DEPTH = 2
GEOMETRY_OPERATIONS = (
    "rotate_90",
    "rotate_180",
    "rotate_270",
    "flip_horizontal",
    "flip_vertical",
    "transpose",
    "anti_diagonal",
)


def grid_shape(grid):
    return len(grid), len(grid[0])


def infer_background(grid):
    counts = {}
    for row in grid:
        for color in row:
            counts[color] = counts.get(color, 0) + 1
    return min(counts, key=lambda color: (-counts[color], color))


def crop_to_content(grid, background=None):
    background = infer_background(grid) if background is None else background
    cells = [
        (row_number, column_number)
        for row_number, row in enumerate(grid)
        for column_number, color in enumerate(row)
        if color != background
    ]
    if not cells:
        return [[background]]
    top = min(row for row, _ in cells)
    bottom = max(row for row, _ in cells)
    left = min(column for _, column in cells)
    right = max(column for _, column in cells)
    return [row[left : right + 1] for row in grid[top : bottom + 1]]


def apply_geometry(grid, operation):
    validate_grid(grid)
    if operation == "rotate_90":
        return [list(row) for row in zip(*grid[::-1], strict=True)]
    if operation == "rotate_180":
        return [row[::-1] for row in grid[::-1]]
    if operation == "rotate_270":
        return [list(row) for row in zip(*grid, strict=True)][::-1]
    if operation == "flip_horizontal":
        return [row[::-1] for row in grid]
    if operation == "flip_vertical":
        return grid[::-1]
    if operation == "transpose":
        return [list(row) for row in zip(*grid, strict=True)]
    if operation == "anti_diagonal":
        return [list(row)[::-1] for row in zip(*grid, strict=True)][::-1]
    raise ValueError(f"unknown geometry operation: {operation}")


def learn_color_map(demos):
    # Color mappings are bijective: distinct source colors cannot collapse.
    mapping = {}
    reverse = {}
    for demo in demos:
        if grid_shape(demo["input"]) != grid_shape(demo["output"]):
            return None
        for input_row, output_row in zip(demo["input"], demo["output"], strict=True):
            for input_color, output_color in zip(input_row, output_row, strict=True):
                if input_color in mapping and mapping[input_color] != output_color:
                    return None
                if output_color in reverse and reverse[output_color] != input_color:
                    return None
                mapping[input_color] = output_color
                reverse[output_color] = input_color
    return mapping


def recolor(grid, mapping):
    return [[mapping.get(color, color) for color in row] for row in grid]


def connected_components(grid, connectivity=4):
    validate_grid(grid)
    if connectivity not in (4, 8):
        raise ValueError("connectivity must be 4 or 8")
    background = infer_background(grid)
    height, width = grid_shape(grid)
    offsets = [(1, 0), (-1, 0), (0, 1), (0, -1)]
    if connectivity == 8:
        offsets += [(1, 1), (1, -1), (-1, 1), (-1, -1)]
    unseen = {
        (row, column)
        for row in range(height)
        for column in range(width)
        if grid[row][column] != background
    }
    components = []
    while unseen:
        start = min(unseen)
        color = grid[start[0]][start[1]]
        stack = [start]
        unseen.remove(start)
        cells = []
        while stack:
            row, column = stack.pop()
            cells.append((row, column))
            for row_delta, column_delta in offsets:
                neighbor = (row + row_delta, column + column_delta)
                if neighbor in unseen and grid[neighbor[0]][neighbor[1]] == color:
                    unseen.remove(neighbor)
                    stack.append(neighbor)
        components.append({"color": color, "cells": sorted(cells)})
    return components


def extract_selected_object(grid, connectivity=4, selection="largest"):
    background = infer_background(grid)
    components = connected_components(grid, connectivity)
    if not components:
        return [[background]]
    if selection == "largest":

        def key(component):
            return (
                -len(component["cells"]),
                min(component["cells"]),
                component["color"],
            )
    elif selection == "smallest":

        def key(component):
            return (
                len(component["cells"]),
                min(component["cells"]),
                component["color"],
            )
    else:
        raise ValueError(f"unknown object selection: {selection}")
    component = min(components, key=key)
    cells = component["cells"]
    top = min(row for row, _ in cells)
    bottom = max(row for row, _ in cells)
    left = min(column for _, column in cells)
    right = max(column for _, column in cells)
    cell_set = set(cells)
    return [
        [
            grid[row][column] if (row, column) in cell_set else background
            for column in range(left, right + 1)
        ]
        for row in range(top, bottom + 1)
    ]


def _relation_for_pair(demo):
    input_height, input_width = grid_shape(demo["input"])
    output_height, output_width = grid_shape(demo["output"])
    if (input_height, input_width) == (output_height, output_width):
        return ("same",)
    if (input_height, input_width) == (output_width, output_height):
        return ("transpose",)
    if output_height % input_height == 0 and output_width % input_width == 0:
        return ("scale", output_height // input_height, output_width // input_width)
    if grid_shape(crop_to_content(demo["input"])) == (output_height, output_width):
        return ("crop",)
    return ("fixed", output_height, output_width)


def infer_shape_relation(demos):
    relations = [_relation_for_pair(demo) for demo in demos]
    return (
        relations[0]
        if all(relation == relations[0] for relation in relations)
        else None
    )


def resize_nearest(grid, height, width):
    if not 1 <= height <= MAX_GRID_SIDE or not 1 <= width <= MAX_GRID_SIDE:
        raise ValidationError("predicted shape exceeds the grid limit")
    input_height, input_width = grid_shape(grid)
    return [
        [
            grid[row * input_height // height][column * input_width // width]
            for column in range(width)
        ]
        for row in range(height)
    ]


def apply_relation(grid, relation):
    kind = relation[0]
    if kind == "same":
        return copy.deepcopy(grid)
    if kind == "transpose":
        return apply_geometry(grid, "transpose")
    if kind == "crop":
        return crop_to_content(grid)
    if kind == "scale":
        input_height, input_width = grid_shape(grid)
        return resize_nearest(
            grid,
            input_height * relation[1],
            input_width * relation[2],
        )
    if kind == "fixed":
        return resize_nearest(grid, relation[1], relation[2])
    raise ValueError(f"unknown shape relation: {relation}")


def complete_symmetry(grid, axis):
    background = infer_background(grid)
    height, width = grid_shape(grid)
    result = copy.deepcopy(grid)
    for row in range(height):
        for column in range(width):
            if axis == "horizontal":
                source = (height - 1 - row, column)
            elif axis == "vertical":
                source = (row, width - 1 - column)
            elif axis == "rotational":
                source = (height - 1 - row, width - 1 - column)
            else:
                raise ValueError(f"unknown symmetry axis: {axis}")
            observed = grid[row][column]
            reflected = grid[source[0]][source[1]]
            if (
                observed != background
                and reflected != background
                and observed != reflected
            ):
                raise ValueError("observed symmetry cells conflict")
            if observed == background and reflected != background:
                result[row][column] = reflected
    return result


def complete_periodic(grid, axis, period):
    # shortcut: repeat only the intact leading template; infer no shifted phase.
    if period < 1:
        raise ValueError("period must be positive")
    background = infer_background(grid)
    height, width = grid_shape(grid)
    result = copy.deepcopy(grid)
    for row in range(height):
        for column in range(width):
            source = (
                (row % period, column)
                if axis == "horizontal"
                else (row, column % period)
            )
            if (
                axis not in ("horizontal", "vertical")
                or source[0] >= height
                or source[1] >= width
            ):
                raise ValueError("period does not fit the grid")
            if (
                grid[row][column] != background
                and grid[row][column] != grid[source[0]][source[1]]
            ):
                raise ValueError("observed periodic cells conflict")
            if (
                grid[row][column] == background
                and grid[source[0]][source[1]] != background
            ):
                result[row][column] = grid[source[0]][source[1]]
    return result


def apply_operation(grid, operation):
    name = operation[0]
    if name in GEOMETRY_OPERATIONS:
        return apply_geometry(grid, name)
    if name == "identity":
        return copy.deepcopy(grid)
    if name == "crop_non_background":
        return crop_to_content(grid)
    if name == "remove_uniform_border":
        height, width = grid_shape(grid)
        if height <= 2 or width <= 2:
            raise ValueError("grid has no removable border")
        border = grid[0][0]
        if any(
            grid[row][column] != border
            for row in range(height)
            for column in range(width)
            if row in (0, height - 1) or column in (0, width - 1)
        ):
            raise ValueError("grid border is not uniform")
        return [row[1:-1] for row in grid[1:-1]]
    if name == "recolor":
        mapping = dict(operation[1])
        if any(
            type(color) is not int or not 0 <= color <= 9 for color in mapping.values()
        ):
            raise ValidationError("recolor mapping contains an invalid color")
        return recolor(grid, mapping)
    if name == "resize_relation":
        return apply_relation(grid, operation[1])
    if name == "extract_object":
        return extract_selected_object(grid, operation[1], operation[2])
    if name == "symmetry":
        return complete_symmetry(grid, operation[1])
    if name == "periodic":
        return complete_periodic(grid, operation[1], operation[2])
    raise ValueError(f"unknown operation: {name}")


def apply_program(grid, program):
    if not 1 <= len(program) <= MAX_COMPOSITION_DEPTH:
        raise ValueError("program depth exceeds the limit")
    result = copy.deepcopy(grid)
    for operation in program:
        result = apply_operation(result, operation)
        validate_grid(result)
    return result


def _append_program(programs, seen, program):
    key = repr(program)
    if key not in seen and len(programs) < MAX_PROGRAMS_PER_TASK:
        seen.add(key)
        programs.append(program)


def _recolor_after_prefix(demos, prefix):
    transformed = []
    for demo in demos:
        try:
            prefix_input = apply_operation(demo["input"], prefix)
        except (ValidationError, ValueError):
            return None
        transformed.append({"input": prefix_input, "output": demo["output"]})
    mapping = learn_color_map(transformed)
    return ("recolor", tuple(sorted(mapping.items()))) if mapping else None


def generate_programs(task):
    demos = task["train"]
    programs = []
    seen = set()
    base = [
        ("identity",),
        *((name,) for name in GEOMETRY_OPERATIONS),
        ("crop_non_background",),
        ("remove_uniform_border",),
    ]
    mapping = learn_color_map(demos)
    if mapping:
        base.append(("recolor", tuple(sorted(mapping.items()))))
    relation = infer_shape_relation(demos)
    if relation and relation[0] in ("transpose", "scale", "crop", "fixed"):
        base.append(("resize_relation", relation))
    for connectivity in (4, 8):
        for selection in ("largest", "smallest"):
            base.append(("extract_object", connectivity, selection))
    for axis in ("horizontal", "vertical", "rotational"):
        base.append(("symmetry", axis))
    for axis in ("horizontal", "vertical"):
        for period in range(1, 7):
            base.append(("periodic", axis, period))
    prefixes = tuple(
        operation
        for operation in base
        if operation[0] not in ("recolor", "symmetry", "periodic")
    )
    for prefix in prefixes:
        recolor_operation = _recolor_after_prefix(demos, prefix)
        if recolor_operation is not None:
            base.append(recolor_operation)
    for operation in base:
        _append_program(programs, seen, (operation,))
    if MAX_COMPOSITION_DEPTH == 2:
        for first in base:
            for second in base:
                _append_program(programs, seen, (first, second))
                if len(programs) >= MAX_PROGRAMS_PER_TASK:
                    # ponytail: the fixed cap limits later depth-two pairs;
                    # depth-1 candidates remain first and deterministic.
                    return programs
    return programs


def _distinct_grids(grids):
    result = []
    seen = set()
    for grid in grids:
        key = tuple(tuple(row) for row in grid)
        if key not in seen:
            seen.add(key)
            result.append(copy.deepcopy(grid))
    return result


def _fallback_grids(task, grid):
    grids = [copy.deepcopy(grid)]
    relation = infer_shape_relation(task["train"])
    if relation:
        try:
            grids.append(apply_relation(grid, relation))
        except (ValidationError, ValueError):
            pass
    mapping = learn_color_map(task["train"])
    if mapping:
        grids.append(recolor(grid, mapping))
    return _distinct_grids(grids)


def solve_task(task, return_diagnostics=False):
    started = time.perf_counter()
    programs = generate_programs(task)
    exact_programs = []
    best_coverage = 0
    for program in programs:
        coverage = 0
        valid = True
        for demo in task["train"]:
            try:
                predicted = apply_program(demo["input"], program)
            except (ValidationError, ValueError):
                valid = False
                continue
            coverage += predicted == demo["output"]
        best_coverage = max(best_coverage, coverage)
        if valid and coverage == len(task["train"]):
            exact_programs.append(program)
    predictions = []
    for test in task["test"]:
        candidates = []
        for program in exact_programs:
            try:
                candidates.append(apply_program(test["input"], program))
            except (ValidationError, ValueError):
                continue
        candidates.extend(_fallback_grids(task, test["input"]))
        candidates = _distinct_grids(candidates)
        if not candidates:
            candidates = [copy.deepcopy(test["input"])]
        if len(candidates) == 1:
            candidates.append(copy.deepcopy(candidates[0]))
        predictions.append({"attempt_1": candidates[0], "attempt_2": candidates[1]})
    diagnostics = {
        "program_count": len(programs),
        "exact_fit": bool(exact_programs),
        "train_demo_coverage": best_coverage / len(task["train"]),
        "elapsed_seconds": time.perf_counter() - started,
    }
    return (predictions, diagnostics) if return_diagnostics else predictions


def symbolic_submission(challenges, return_diagnostics=False):
    validate_challenges(challenges)
    submission = {}
    diagnostics = {}
    for task_id, task in challenges.items():
        predictions, task_diagnostics = solve_task(task, return_diagnostics=True)
        submission[task_id] = predictions
        diagnostics[task_id] = task_diagnostics
    return (submission, diagnostics) if return_diagnostics else submission


def build_submission(challenges):
    solver_mode = os.environ.get("ARC_SOLVER_MODE", "symbolic").strip().lower()
    if solver_mode == "identity":
        return identity_submission(challenges)
    if solver_mode != "symbolic":
        raise ValueError("ARC_SOLVER_MODE must be 'symbolic' or 'identity'")
    return symbolic_submission(challenges)


TRAIN_HOLDOUT_MODULUS = 5
TRAIN_HOLDOUT_BUCKET = 0


def training_split(task_id):
    digest = hashlib.sha256(task_id.encode("ascii")).digest()
    return (
        "heldout"
        if int.from_bytes(digest[:4], "big") % TRAIN_HOLDOUT_MODULUS
        == TRAIN_HOLDOUT_BUCKET
        else "iteration"
    )


def _score_training_split(challenges, solutions, submission, split):
    selected_ids = {
        task_id for task_id in challenges if training_split(task_id) == split
    }
    selected_challenges = {task_id: challenges[task_id] for task_id in selected_ids}
    selected_solutions = {task_id: solutions[task_id] for task_id in selected_ids}
    if not selected_challenges:
        return None
    return score_submission(
        selected_challenges,
        selected_solutions,
        {task_id: submission[task_id] for task_id in selected_ids},
    )


def evaluate_training(challenges, solutions):
    validate_challenges(challenges)
    validate_solutions(challenges, solutions)
    started = time.perf_counter()
    training_submission = {}
    diagnostics = {}
    for task_id, task in challenges.items():
        predictions, task_diagnostics = solve_task(task, return_diagnostics=True)
        training_submission[task_id] = predictions
        diagnostics[task_id] = task_diagnostics
    validate_submission(challenges, training_submission)
    score = score_submission(challenges, solutions, training_submission)
    failures = []
    for task_id, _task in challenges.items():
        for test_index, expected in enumerate(solutions[task_id]):
            prediction = training_submission[task_id][test_index]
            if any(prediction[key] == expected for key in ("attempt_1", "attempt_2")):
                continue
            category = (
                "wrong_shape"
                if all(
                    grid_shape(prediction[key]) != grid_shape(expected)
                    for key in ("attempt_1", "attempt_2")
                )
                else "content_or_placement"
            )
            if len(failures) < 5:
                failures.append(
                    {
                        "task_id": task_id,
                        "test_index": test_index,
                        "category": category,
                        "train_demo_coverage": diagnostics[task_id][
                            "train_demo_coverage"
                        ],
                    }
                )
    exact_fit_tasks = sum(
        task_diagnostics["exact_fit"] for task_diagnostics in diagnostics.values()
    )
    task_times = [
        task_diagnostics["elapsed_seconds"] for task_diagnostics in diagnostics.values()
    ]
    metrics = {
        **score,
        "exact_fit_tasks": exact_fit_tasks,
        "candidate_coverage": exact_fit_tasks / len(challenges),
        "train_demo_coverage": sum(
            task_diagnostics["train_demo_coverage"]
            for task_diagnostics in diagnostics.values()
        )
        / len(challenges),
        "total_task_runtime_seconds": sum(task_times),
        "max_task_runtime_seconds": max(task_times),
        "wall_clock_seconds": time.perf_counter() - started,
        "representative_failures": failures,
        "split_metrics": {
            split: _score_training_split(
                challenges, solutions, training_submission, split
            )
            for split in ("iteration", "heldout")
        },
    }
    return training_submission, metrics


def identity_submission(challenges):
    validate_challenges(challenges)
    return {
        task_id: [
            {
                "attempt_1": copy.deepcopy(test["input"]),
                "attempt_2": copy.deepcopy(test["input"]),
            }
            for test in task["test"]
        ]
        for task_id, task in challenges.items()
    }


def score_submission(challenges, solutions, submission):
    validate_submission(challenges, submission)
    validate_solutions(challenges, solutions)
    correct_outputs = 0
    solved_tasks = 0
    task_pass_fractions = []
    total_outputs = 0
    for task_id, task in challenges.items():
        task_correct = 0
        for prediction, expected in zip(
            submission[task_id], solutions[task_id], strict=True
        ):
            if (
                prediction["attempt_1"] == expected
                or prediction["attempt_2"] == expected
            ):
                task_correct += 1
        task_outputs = len(task["test"])
        correct_outputs += task_correct
        total_outputs += task_outputs
        solved_tasks += task_correct == task_outputs
        task_pass_fractions.append(task_correct / task_outputs)
    output_accuracy = correct_outputs / total_outputs
    task_solve_rate = solved_tasks / len(challenges)
    task_averaged_pass_at_2 = sum(task_pass_fractions) / len(task_pass_fractions)
    return {
        "correct_outputs": correct_outputs,
        "total_outputs": total_outputs,
        "output_accuracy": output_accuracy,
        "official_score": output_accuracy,
        "solved_tasks": solved_tasks,
        "total_tasks": len(challenges),
        "task_solve_rate": task_solve_rate,
        "task_averaged_pass_at_2": task_averaged_pass_at_2,
    }


def read_json(path):
    with Path(path).open(encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(value, handle, indent=2, sort_keys=True)
        handle.write(chr(10))


def resolve_repo_root():
    cwd = Path.cwd()
    if (cwd / "data").is_dir():
        return cwd
    if cwd.name == "notebooks" and (cwd.parent / "data").is_dir():
        return cwd.parent
    return cwd


def resolve_data_dir():
    configured = os.environ.get("ARC_DATA_DIR")
    if configured:
        return Path(configured)
    candidates = [
        Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2"),
        Path("/kaggle/input/arc-prize-2026-arc-agi-2"),
    ]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        candidates.extend(
            path for path in sorted(kaggle_input.iterdir()) if path.is_dir()
        )
    candidates.append(resolve_repo_root() / "data")
    for candidate in candidates:
        if (candidate / "arc-agi_test_challenges.json").exists() or (
            candidate / "arc-agi_evaluation_challenges.json"
        ).exists():
            return candidate
    raise FileNotFoundError("could not locate ARC-AGI-2 challenge data")


def load_inputs(mode, data_dir):
    if mode in ("evaluation", "training"):
        prefix = "evaluation" if mode == "evaluation" else "training"
        challenges = read_json(Path(data_dir) / f"arc-agi_{prefix}_challenges.json")
        validate_challenges(challenges)
        solutions = read_json(Path(data_dir) / f"arc-agi_{prefix}_solutions.json")
        validate_solutions(challenges, solutions)
        return challenges, solutions
    if mode != "submission":
        raise ValueError("ARC_MODE must be 'submission', 'evaluation', or 'training'")
    challenges = read_json(Path(data_dir) / "arc-agi_test_challenges.json")
    validate_challenges(challenges)
    return challenges, None


def resolve_output_path():
    configured = os.environ.get("ARC_OUTPUT_PATH")
    if configured:
        return Path(configured)
    kaggle_working = Path("/kaggle/working")
    if kaggle_working.is_dir():
        return kaggle_working / "submission.json"
    return resolve_repo_root() / "artifacts" / "submission.json"

## Run mode

Kaggle uses submission mode by default. For local evaluation, set `ARC_MODE=evaluation`; for a local placeholder submission, set `ARC_MODE=submission`. `ARC_DATA_DIR` and `ARC_OUTPUT_PATH` are optional overrides for local execution.

In [ ]:
mode = os.environ.get("ARC_MODE", "submission").strip().lower()
data_dir = resolve_data_dir()
started = time.perf_counter()
challenges, solutions = load_inputs(mode, data_dir)
if mode == "training":
    submission, training_metrics = evaluate_training(challenges, solutions)
else:
    submission = build_submission(challenges)
validate_submission(challenges, submission)
output_path = resolve_output_path()
write_json(output_path, submission)
round_trip = read_json(output_path)
validate_submission(challenges, round_trip)
elapsed = time.perf_counter() - started
total_outputs = sum(len(task["test"]) for task in challenges.values())
print(f"mode: {mode}")
print(f"data directory: {data_dir}")
print(f"tasks: {len(challenges)}; test outputs: {total_outputs}")
print(f"validated JSON readback: {output_path}")
if mode == "evaluation":
    metrics = score_submission(challenges, solutions, round_trip)
    print(
        f"output accuracy: {metrics['correct_outputs']}/{metrics['total_outputs']} = "
        f"{metrics['output_accuracy']:.6f}"
    )
    print(
        f"task solve rate: {metrics['solved_tasks']}/{metrics['total_tasks']} = "
        f"{metrics['task_solve_rate']:.6f}"
    )
    print(f"task-averaged pass@2: {metrics['task_averaged_pass_at_2']:.6f}")
    print(f"official score (output-weighted): {metrics['official_score']:.6f}")
elif mode == "training":
    print("training metrics: " + json.dumps(training_metrics, sort_keys=True))
else:
    print("solutions loaded: no")
print(f"solver mode: {os.environ.get('ARC_SOLVER_MODE', 'symbolic').strip().lower()}")
print(f"elapsed seconds: {elapsed:.6f}")